# Cats vs Dogs: comparison of 5 pretrained models

This notebook trains and compares several image classifiers on the Kaggle *Cats vs Dogs* dataset and exports
the best one for the Flask backend used by the web, desktop and mobile clients.

**Steps**
1. Download the dataset and remove corrupted images.
2. **Balance the classes**: same number of cats and dogs.
3. **Split** into train (70%), validation (15%) and test (15%) and save the splits as CSV files.
4. Train a **baseline CNN from scratch** as a reference.
5. Train **5 pretrained models** with **transfer learning + fine-tuning**:
   ResNet50, DenseNet121, MobileNetV2, EfficientNetB0 and Xception.
6. Evaluate every model on the test set: **accuracy, precision, recall, specificity, F1, AUC**, confusion matrices,
   ROC curves, size and speed.
7. **K-fold cross-validation** on the best model to check that its result is stable.
8. Export the best model.

**Before running:** `Runtime -> Change runtime type -> T4 GPU`, then `Runtime -> Run all`.
Results are saved to Google Drive after every model, so if Colab disconnects just run everything again:
finished models are skipped and training continues where it stopped.

In [ ]:
import gc
import glob
import json
import pathlib
import random
import shutil
import time
import urllib.request
import zipfile

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
import keras
from keras import layers
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix, roc_auc_score, roc_curve
from sklearn.model_selection import StratifiedKFold

print("TensorFlow:", tf.__version__)
print("Keras:", keras.__version__)
print("GPU:", tf.config.list_physical_devices("GPU") or "none (training will be very slow!)")

## 1. Google Drive and configuration

Everything important (splits, checkpoints, logs, results, figures) is stored in Google Drive under
`MyDrive/cats_dogs_vision/`, so nothing is lost if the Colab session ends.

In [ ]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
    WORK_DIR = pathlib.Path("/content/drive/MyDrive/cats_dogs_vision")
except ImportError:  # running outside Colab
    WORK_DIR = pathlib.Path("work")

SPLITS_DIR = WORK_DIR / "splits"
MODELS_DIR = WORK_DIR / "models"
LOGS_DIR = WORK_DIR / "logs"
RESULTS_DIR = WORK_DIR / "results"
FIG_DIR = WORK_DIR / "figures"
EXPORT_DIR = WORK_DIR / "export"
for d in (SPLITS_DIR, MODELS_DIR, LOGS_DIR, RESULTS_DIR, FIG_DIR, EXPORT_DIR):
    d.mkdir(parents=True, exist_ok=True)
print("Saving everything to", WORK_DIR)

In [ ]:
IMG_SIZE = 180                    # same as the class example
INPUT_SHAPE = (IMG_SIZE, IMG_SIZE, 3)
BATCH_SIZE = 32
SEED = 1337

BASELINE_EPOCHS = 20              # model from scratch (early stopping usually ends it sooner)
HEAD_EPOCHS = 5                   # transfer learning, phase 1: only the new classifier is trained
FINE_TUNE_EPOCHS = 5              # transfer learning, phase 2: top layers of the pretrained model are unfrozen
FINE_TUNE_FRACTION = 0.2          # fraction of the pretrained layers unfrozen in phase 2

KFOLDS = 5                        # cross-validation folds
KFOLD_HEAD_EPOCHS = 3
KFOLD_FINE_TUNE_EPOCHS = 2

CLASS_NAMES = ["Cat", "Dog"]      # label 0 = Cat, label 1 = Dog (the positive class)
DATA_DIR = pathlib.Path("PetImages")
AUTOTUNE = tf.data.AUTOTUNE

keras.utils.set_random_seed(SEED)

## 2. Download the dataset and remove corrupted images

Same dataset as the class example. Some files are corrupted or are not real JPEGs (no `JFIF` header),
so they are deleted.

In [ ]:
URL = "https://download.microsoft.com/download/3/E/1/3E1C3F21-ECDB-4869-8368-6DEBA77B919F/kagglecatsanddogs_5340.zip"

if not DATA_DIR.exists():
    print("Downloading (786 MB)...")
    urllib.request.urlretrieve(URL, "kagglecatsanddogs_5340.zip")
    with zipfile.ZipFile("kagglecatsanddogs_5340.zip") as zf:
        zf.extractall(".")

num_skipped = 0
for folder_name in CLASS_NAMES:
    for path in (DATA_DIR / folder_name).iterdir():
        with open(path, "rb") as fobj:
            is_jfif = b"JFIF" in fobj.peek(10)
        if not is_jfif:
            num_skipped += 1
            path.unlink()  # delete corrupted image
print(f"Deleted {num_skipped} images.")

## 3. Class balancing

The dataset has slightly more cats than dogs. To avoid a model that is biased towards the bigger class,
both lists are shuffled and cut to the size of the smaller one (*undersampling*), like in the class example.
A fixed seed makes the shuffle reproducible.

In [ ]:
cat_files = sorted(glob.glob(str(DATA_DIR / "Cat" / "*.jpg")))
dog_files = sorted(glob.glob(str(DATA_DIR / "Dog" / "*.jpg")))

rng = random.Random(SEED)
rng.shuffle(cat_files)
rng.shuffle(dog_files)

num_cats, num_dogs = len(cat_files), len(dog_files)
num_samples = min(num_cats, num_dogs)
cats = cat_files[:num_samples]
dogs = dog_files[:num_samples]

print(f"Before balancing: {num_cats} cats, {num_dogs} dogs")
print(f"After balancing:  {len(cats)} cats, {len(dogs)} dogs")

## 4. Train (70%) / validation (15%) / test (15%) split

Each class is split separately, so every split stays balanced. The result is saved as `train.csv`, `val.csv`
and `test.csv` (columns `filename`, `label`), like in the class example.

- **train**: used to learn the weights.
- **validation**: used during training (callbacks) and to **choose the best model**.
- **test**: used only once at the end, to report the final metrics on images no model has seen.

If the CSV files already exist in Drive they are reused, so every run (and the cross-validation) uses exactly
the same split.

In [ ]:
def split_70_15_15(files):
    n = len(files)
    a, b = int(n * 0.70), int(n * 0.85)
    return files[:a], files[a:b], files[b:]


def make_df(cat_list, dog_list):
    df = pd.concat([
        pd.DataFrame({"filename": cat_list, "label": 0}),
        pd.DataFrame({"filename": dog_list, "label": 1}),
    ])
    return df.sample(frac=1.0, random_state=SEED).reset_index(drop=True)  # shuffle


if (SPLITS_DIR / "test.csv").exists():
    df_train, df_val, df_test = (pd.read_csv(SPLITS_DIR / f"{s}.csv") for s in ("train", "val", "test"))
    print("Loaded existing splits from Drive")
else:
    train_cats, val_cats, test_cats = split_70_15_15(cats)
    train_dogs, val_dogs, test_dogs = split_70_15_15(dogs)
    df_train = make_df(train_cats, train_dogs)
    df_val = make_df(val_cats, val_dogs)
    df_test = make_df(test_cats, test_dogs)
    for name, df in (("train", df_train), ("val", df_val), ("test", df_test)):
        df.to_csv(SPLITS_DIR / f"{name}.csv", index=False)
    print("Created and saved new splits")

split_counts = pd.DataFrame({
    name: df["label"].value_counts().sort_index().rename(index=dict(enumerate(CLASS_NAMES)))
    for name, df in (("train", df_train), ("validation", df_val), ("test", df_test))
}).T
split_counts["total"] = split_counts.sum(axis=1)
split_counts["% of data"] = (100 * split_counts["total"] / split_counts["total"].sum()).round(1)
split_counts

In [ ]:
split_counts[CLASS_NAMES].plot.bar(figsize=(7, 4), rot=0, color=["#888888", "#333333"])
plt.title("Images per split and class (balanced)")
plt.ylabel("images")
plt.savefig(FIG_DIR / "class_balance.png", bbox_inches="tight")
plt.show()

## 5. tf.data pipelines

`process_path` reads each file, decodes the JPEG and resizes it to 180x180 (like in the class notes).
Pixels are kept in **[0, 255]**: each pretrained model needs a different normalization, so that step is
**inside each model** (section 7). This also means the backend only has to resize the image.

The decoded images are **cached in memory** as `uint8`, so the JPEGs are decoded only once per session and
the following epochs (and the following models) are much faster.

> During training you may see the warning *"Your input ran out of data; interrupting training"* once per
> training run. It is harmless: Keras does not know the number of batches in advance (because
> `ignore_errors()` may skip unreadable images), so it detects the end of the first epoch by running out of data.

In [ ]:
def process_path(filename, label):
    img = tf.io.read_file(filename)
    img = tf.io.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
    img = tf.cast(tf.round(img), tf.uint8)  # uint8 for the in-memory cache (4x smaller than float32)
    return img, label


def to_float(images, labels):
    return tf.cast(images, tf.float32), labels


def make_dataset(df, training):
    ds = tf.data.Dataset.from_tensor_slices((df["filename"].values, df["label"].values.astype("float32")))
    ds = ds.map(process_path, num_parallel_calls=AUTOTUNE).ignore_errors().cache()
    if training:
        ds = ds.shuffle(4096, seed=SEED)
    return ds.batch(BATCH_SIZE).map(to_float, num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)


train_ds = make_dataset(df_train, training=True)
val_ds = make_dataset(df_val, training=False)
test_ds = make_dataset(df_test, training=False)

# True labels in dataset order (validation and test are not shuffled, so the order is always the same).
# This first pass also fills the cache.
y_val = np.concatenate([y.numpy() for _, y in val_ds]).astype(int)
y_test = np.concatenate([y.numpy() for _, y in test_ds]).astype(int)
np.save(RESULTS_DIR / "y_test.npy", y_test)
print(f"validation: {len(y_val)} images, test: {len(y_test)} images")

In [ ]:
images, labels = next(iter(train_ds))
plt.figure(figsize=(10, 10))
for i in range(min(9, len(images))):
    plt.subplot(3, 3, i + 1)
    plt.imshow(images[i].numpy().astype("uint8"))
    plt.title(CLASS_NAMES[int(labels[i])])
    plt.axis("off")
plt.suptitle("Training samples")
plt.savefig(FIG_DIR / "samples.png", bbox_inches="tight")
plt.show()

## 6. Data augmentation

Random flips, rotations and zooms create variations of the training images to reduce overfitting.
These layers are part of the models but are **only active during training**.

In [ ]:
def make_augmentation():
    return keras.Sequential([
        layers.RandomFlip("horizontal"),
        layers.RandomRotation(0.1),
        layers.RandomZoom(0.1),
    ], name="augmentation")


aug = make_augmentation()
plt.figure(figsize=(10, 10))
for i in range(9):
    plt.subplot(3, 3, i + 1)
    plt.imshow(aug(images[:1], training=True)[0].numpy().astype("uint8"))
    plt.axis("off")
plt.suptitle("Same image, different augmentations")
plt.savefig(FIG_DIR / "augmentation.png", bbox_inches="tight")
plt.show()

## 7. Models

### Why `include_top=False`
All 5 models were pretrained on **ImageNet** (1.4M images, 1000 classes). With `include_top=True` and
`weights="imagenet"` Keras only allows `classes=1000`, because the pretrained weights include the original
1000-class output layer. For transfer learning we load the models **without that top layer**
(`include_top=False`) and add our own classifier: `GlobalAveragePooling2D -> Dropout -> Dense(1, sigmoid)`.
The sigmoid output is the probability that the image is a **dog**; `binary_crossentropy` is the matching loss.

### Input preprocessing (inside each model)
Each architecture was trained on ImageNet with a different input normalization. To make the exported model
work with plain [0, 255] RGB images, the normalization is the first layer of each model, built only with
standard Keras layers (so the backend can load any of them without extra code):

| Model | Keras `preprocess_input` mode | Layer we use |
|---|---|---|
| ResNet50 | `caffe`: RGB->BGR, subtract ImageNet mean | frozen 1x1 `Conv2D` (swaps channels, bias = -mean) |
| DenseNet121 | `torch`: /255, subtract mean, divide by std | `Normalization(mean, variance)` |
| MobileNetV2 | `tf`: scale to [-1, 1] | `Rescaling(1/127.5, offset=-1)` |
| EfficientNetB0 | none (normalization already inside the model) | nothing |
| Xception | `tf`: scale to [-1, 1] | `Rescaling(1/127.5, offset=-1)` |

The next cell **checks** that every layer gives exactly the same result as the official `preprocess_input`.

In [ ]:
def preprocess_caffe(x):
    # 1x1 convolution whose kernel is a channel permutation (RGB -> BGR) and whose bias is -mean.
    conv = layers.Conv2D(3, 1, trainable=False, name="preprocess_caffe")
    out = conv(x)
    kernel = np.zeros((1, 1, 3, 3), dtype="float32")
    kernel[0, 0, 2, 0] = kernel[0, 0, 1, 1] = kernel[0, 0, 0, 2] = 1.0
    conv.set_weights([kernel, -np.array([103.939, 116.779, 123.68], dtype="float32")])
    return out


def preprocess_torch(x):
    mean = np.array([0.485, 0.456, 0.406]) * 255
    std = np.array([0.229, 0.224, 0.225]) * 255
    return layers.Normalization(mean=mean.tolist(), variance=(std ** 2).tolist(), name="preprocess_torch")(x)


def preprocess_tf(x):
    return layers.Rescaling(1.0 / 127.5, offset=-1, name="preprocess_tf")(x)


def preprocess_none(x):
    return layers.Identity(name="preprocess_none")(x)  # EfficientNet normalizes internally


PRETRAINED = {
    "ResNet50": (keras.applications.ResNet50, preprocess_caffe, keras.applications.resnet50.preprocess_input),
    "DenseNet121": (keras.applications.DenseNet121, preprocess_torch, keras.applications.densenet.preprocess_input),
    "MobileNetV2": (keras.applications.MobileNetV2, preprocess_tf, keras.applications.mobilenet_v2.preprocess_input),
    "EfficientNetB0": (keras.applications.EfficientNetB0, preprocess_none, keras.applications.efficientnet.preprocess_input),
    "Xception": (keras.applications.Xception, preprocess_tf, keras.applications.xception.preprocess_input),
}

# Check: our preprocessing layers == official preprocess_input
sample = np.random.default_rng(SEED).uniform(0, 255, size=(2, IMG_SIZE, IMG_SIZE, 3)).astype("float32")
for name, (_, preprocess, reference) in PRETRAINED.items():
    inp = keras.Input(INPUT_SHAPE)
    ours = keras.Model(inp, preprocess(inp))(sample).numpy()
    official = np.asarray(reference(sample.copy()))
    print(f"{name:15s} max difference: {np.abs(ours - official).max():.6f}")
    assert np.allclose(ours, official, atol=1e-3), name

In [ ]:
def build_baseline():
    """Small CNN trained from scratch (reference model)."""
    inputs = keras.Input(INPUT_SHAPE, name="image")
    x = make_augmentation()(inputs)
    x = layers.Rescaling(1.0 / 255)(x)
    for filters in (32, 64, 128, 256):
        x = layers.Conv2D(filters, 3, padding="same", activation="relu")(x)
        x = layers.BatchNormalization()(x)
        x = layers.MaxPooling2D()(x)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(1, activation="sigmoid", name="dog_probability")(x)
    return keras.Model(inputs, outputs, name="BaselineCNN"), None


def build_pretrained(name):
    """Pretrained model without its ImageNet top + our classifier. Returns (model, base)."""
    constructor, preprocess, _ = PRETRAINED[name]
    base = constructor(include_top=False, weights="imagenet", input_shape=INPUT_SHAPE)
    base.trainable = False  # phase 1: frozen

    inputs = keras.Input(INPUT_SHAPE, name="image")
    x = make_augmentation()(inputs)
    x = preprocess(x)
    x = base(x, training=False)  # keep BatchNormalization layers in inference mode, also when fine-tuning
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(1, activation="sigmoid", name="dog_probability")(x)
    return keras.Model(inputs, outputs, name=name), base


def unfreeze_top(base, fraction):
    """Phase 2: unfreeze the last `fraction` of the pretrained layers (BatchNormalization stays frozen)."""
    n = max(1, int(len(base.layers) * fraction))
    base.trainable = True
    for layer in base.layers[:-n]:
        layer.trainable = False
    for layer in base.layers[-n:]:
        if isinstance(layer, layers.BatchNormalization):
            layer.trainable = False
    return n


MODEL_SPECS = {"BaselineCNN": {"build": build_baseline, "type": "from scratch"}}
for _name in PRETRAINED:
    MODEL_SPECS[_name] = {"build": lambda n=_name: build_pretrained(n), "type": "pretrained (ImageNet)"}

for name, spec in MODEL_SPECS.items():
    m, base = spec["build"]()
    print(f"{name:15s} {m.count_params() / 1e6:6.2f} M parameters"
          + (f", {len(base.layers)} pretrained layers" if base is not None else ""))
    del m, base
keras.backend.clear_session()

## 8. Training

### Callbacks
| Callback | What it does |
|---|---|
| `ModelCheckpoint` | Saves the model to Drive every time the **validation loss** improves (keeps the best epoch, not the last one) |
| `EarlyStopping` | Stops training when the validation loss has not improved for 3 epochs (avoids overfitting and saves time) |
| `ReduceLROnPlateau` | Divides the learning rate by 5 when the validation loss stops improving |
| `CSVLogger` | Writes the metrics of every epoch to a CSV file (used for the training curves) |

### Transfer learning in two phases
1. **Phase 1, feature extraction**: the pretrained model is frozen; only the new classifier learns (lr = 1e-3).
2. **Phase 2, fine-tuning**: the top 20% of the pretrained layers are unfrozen and trained with a very small
   learning rate (1e-5) so the ImageNet knowledge is adapted, not destroyed.

The baseline CNN has no pretrained part, so it trains in a single phase.

In [ ]:
def compile_model(model, learning_rate):
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate),
        loss="binary_crossentropy",
        metrics=[
            keras.metrics.BinaryAccuracy(name="accuracy"),
            keras.metrics.Precision(name="precision"),
            keras.metrics.Recall(name="recall"),
            keras.metrics.AUC(name="auc"),
        ],
    )


def make_callbacks(checkpoint_path, log_path, best_val_loss=None, append=False):
    return [
        keras.callbacks.ModelCheckpoint(str(checkpoint_path), monitor="val_loss", save_best_only=True,
                                        initial_value_threshold=best_val_loss, verbose=1),
        keras.callbacks.EarlyStopping(monitor="val_loss", patience=3),
        keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.2, patience=2, min_lr=1e-7, verbose=1),
        keras.callbacks.CSVLogger(str(log_path), append=append),
    ]


def compute_metrics(y_true, y_prob, threshold=0.5):
    """Binary metrics with Dog (1) as the positive class."""
    y_pred = (y_prob >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    return {
        "accuracy": (tp + tn) / len(y_true),
        "precision": precision,
        "recall": recall,                                   # = sensitivity, true positive rate
        "specificity": tn / (tn + fp) if tn + fp else 0.0,  # = true negative rate
        "f1": 2 * precision * recall / (precision + recall) if precision + recall else 0.0,
        "auc": roc_auc_score(y_true, y_prob),
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
    }


def inference_ms(model, runs=30):
    x = tf.zeros((1,) + INPUT_SHAPE)
    for _ in range(5):
        model(x, training=False)  # warm-up
    start = time.perf_counter()
    for _ in range(runs):
        model(x, training=False).numpy()
    return (time.perf_counter() - start) / runs * 1000


def train_and_evaluate(name):
    result_file = RESULTS_DIR / f"{name}.json"
    if result_file.exists():
        print(f"{name}: already trained, loading saved results")
        return json.loads(result_file.read_text())

    checkpoint = MODELS_DIR / f"{name}.keras"
    log = LOGS_DIR / f"{name}.csv"
    checkpoint.unlink(missing_ok=True)  # leftovers from an interrupted run
    log.unlink(missing_ok=True)

    keras.backend.clear_session()
    gc.collect()
    spec = MODEL_SPECS[name]
    model, base = spec["build"]()
    start = time.time()

    print(f"===== {name}: phase 1 =====")
    compile_model(model, 1e-3)
    epochs_1 = BASELINE_EPOCHS if base is None else HEAD_EPOCHS
    h1 = model.fit(train_ds, validation_data=val_ds, epochs=epochs_1, callbacks=make_callbacks(checkpoint, log))
    head_epochs = len(h1.epoch)

    unfrozen = 0
    if base is not None:
        print(f"===== {name}: phase 2 (fine-tuning) =====")
        unfrozen = unfreeze_top(base, FINE_TUNE_FRACTION)
        compile_model(model, 1e-5)
        model.fit(train_ds, validation_data=val_ds, epochs=head_epochs + FINE_TUNE_EPOCHS, initial_epoch=head_epochs,
                  callbacks=make_callbacks(checkpoint, log, best_val_loss=min(h1.history["val_loss"]), append=True))
    train_minutes = (time.time() - start) / 60

    # Evaluate the best checkpoint (best validation loss), not the last epoch.
    best = keras.models.load_model(checkpoint)
    val_prob = best.predict(val_ds, verbose=0).ravel()
    test_prob = best.predict(test_ds, verbose=0).ravel()
    np.save(RESULTS_DIR / f"{name}_test_prob.npy", test_prob)

    result = {
        "model": name,
        "type": spec["type"],
        "params_millions": round(best.count_params() / 1e6, 2),
        "size_mb": round(checkpoint.stat().st_size / 1e6, 1),
        "train_minutes": round(train_minutes, 1),
        "epochs": len(pd.read_csv(log)),
        "head_epochs": head_epochs,
        "unfrozen_layers": unfrozen,
        "inference_ms": round(inference_ms(best), 1),
        "validation": compute_metrics(y_val, val_prob),
        "test": compute_metrics(y_test, test_prob),
    }
    result_file.write_text(json.dumps(result, indent=2))
    print(json.dumps({k: round(v, 4) for k, v in result["test"].items()}, indent=2))
    return result

### 8.1 Baseline CNN from scratch
The first model also fills the image cache, so its first epoch is slower.

In [ ]:
results = {"BaselineCNN": train_and_evaluate("BaselineCNN")}

### 8.2 ResNet50

In [ ]:
results["ResNet50"] = train_and_evaluate("ResNet50")

### 8.3 DenseNet121

In [ ]:
results["DenseNet121"] = train_and_evaluate("DenseNet121")

### 8.4 MobileNetV2

In [ ]:
results["MobileNetV2"] = train_and_evaluate("MobileNetV2")

### 8.5 EfficientNetB0

In [ ]:
results["EfficientNetB0"] = train_and_evaluate("EfficientNetB0")

### 8.6 Xception

In [ ]:
results["Xception"] = train_and_evaluate("Xception")

## 9. Comparison

### Metrics (Dog = positive class)

| | Predicted Cat | Predicted Dog |
|---|---|---|
| **Real Cat** | TN (true negative) | FP (false positive) |
| **Real Dog** | FN (false negative) | TP (true positive) |

| Metric | Formula | Meaning |
|---|---|---|
| **Accuracy** | (TP + TN) / total | % of all images classified correctly |
| **Precision** | TP / (TP + FP) | When it says "dog", how often it is really a dog |
| **Recall** (sensitivity) | TP / (TP + FN) | % of the real dogs that it finds |
| **Specificity** | TN / (TN + FP) | % of the real cats that it recognizes as cats |
| **F1** | 2 * P * R / (P + R) | Balance between precision and recall |
| **AUC** | area under the ROC curve | How well it separates the classes at every threshold (1.0 = perfect) |

The table is sorted by **validation F1**, which is the criterion used to choose the best model.
The test columns are the final, unbiased results.

In [ ]:
rows = []
for name, r in results.items():
    rows.append({
        "Model": name,
        "Type": r["type"],
        "Params (M)": r["params_millions"],
        "Size (MB)": r["size_mb"],
        "Train (min)": r["train_minutes"],
        "Epochs": r["epochs"],
        "Inference (ms)": r["inference_ms"],
        "Val F1": r["validation"]["f1"],
        **{f"Test {m}": r["test"][m] for m in ("accuracy", "precision", "recall", "specificity", "f1", "auc")},
    })
comparison = pd.DataFrame(rows).sort_values("Val F1", ascending=False).reset_index(drop=True)
comparison.to_csv(RESULTS_DIR / "comparison.csv", index=False)

BEST = comparison.loc[0, "Model"]
print("Best model (highest validation F1):", BEST)
comparison.style.format(precision=4).highlight_max(
    subset=[c for c in comparison.columns if c.startswith(("Val", "Test"))], color="#dddddd")

In [ ]:
metric_names = ["accuracy", "precision", "recall", "specificity", "f1", "auc"]
plot_df = comparison.set_index("Model")[[f"Test {m}" for m in metric_names]]
plot_df.columns = metric_names
ax = plot_df.T.plot.bar(figsize=(13, 5), rot=0, width=0.8)
ax.set_ylim(max(0.0, plot_df.values.min() - 0.05), 1.0)
ax.set_title("Test metrics per model")
ax.legend(ncol=3, loc="lower right")
plt.savefig(FIG_DIR / "metrics_comparison.png", bbox_inches="tight")
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
for ax, name in zip(axes.ravel(), comparison["Model"]):
    t = results[name]["test"]
    cm = np.array([[t["tn"], t["fp"]], [t["fn"], t["tp"]]])
    ConfusionMatrixDisplay(cm, display_labels=CLASS_NAMES).plot(ax=ax, colorbar=False, cmap="Greys")
    ax.set_title(f"{name}\naccuracy {t['accuracy']:.2%}")
plt.tight_layout()
plt.savefig(FIG_DIR / "confusion_matrices.png", bbox_inches="tight")
plt.show()

In [ ]:
plt.figure(figsize=(7, 6))
for name in comparison["Model"]:
    prob = np.load(RESULTS_DIR / f"{name}_test_prob.npy")
    fpr, tpr, _ = roc_curve(y_test, prob)
    plt.plot(fpr, tpr, label=f"{name} (AUC {results[name]['test']['auc']:.4f})")
plt.plot([0, 1], [0, 1], "k--", linewidth=1, label="random")
plt.xlabel("False positive rate (1 - specificity)")
plt.ylabel("True positive rate (recall)")
plt.title("ROC curves on the test set")
plt.legend()
plt.savefig(FIG_DIR / "roc_curves.png", bbox_inches="tight")
plt.show()

# Zoom on the top-left corner, where the pretrained models are
plt.figure(figsize=(7, 6))
for name in comparison["Model"]:
    fpr, tpr, _ = roc_curve(y_test, np.load(RESULTS_DIR / f"{name}_test_prob.npy"))
    plt.plot(fpr, tpr, label=name)
plt.xlim(0, 0.15)
plt.ylim(0.85, 1.0)
plt.xlabel("False positive rate")
plt.ylabel("True positive rate")
plt.title("ROC curves (zoom)")
plt.legend()
plt.savefig(FIG_DIR / "roc_curves_zoom.png", bbox_inches="tight")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
for name in comparison["Model"]:
    log = pd.read_csv(LOGS_DIR / f"{name}.csv")
    axes[0].plot(log["epoch"] + 1, log["val_accuracy"], marker=".", label=name)
    axes[1].plot(log["epoch"] + 1, log["val_loss"], marker=".", label=name)
axes[0].set_title("Validation accuracy")
axes[1].set_title("Validation loss")
for ax in axes:
    ax.set_xlabel("epoch")
    ax.legend()
plt.savefig(FIG_DIR / "validation_curves.png", bbox_inches="tight")
plt.show()

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
for ax, name in zip(axes.ravel(), comparison["Model"]):
    log = pd.read_csv(LOGS_DIR / f"{name}.csv")
    ax.plot(log["epoch"] + 1, log["accuracy"], label="train")
    ax.plot(log["epoch"] + 1, log["val_accuracy"], label="validation")
    if results[name]["unfrozen_layers"]:
        ax.axvline(results[name]["head_epochs"] + 0.5, color="gray", linestyle="--", label="start fine-tuning")
    ax.set_title(name)
    ax.set_xlabel("epoch")
    ax.legend()
plt.suptitle("Train vs validation accuracy")
plt.tight_layout()
plt.savefig(FIG_DIR / "training_curves.png", bbox_inches="tight")
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
for _, row in comparison.iterrows():
    plt.scatter(row["Inference (ms)"], row["Test accuracy"], s=row["Size (MB)"] * 8 + 30, alpha=0.6)
    plt.annotate(row["Model"], (row["Inference (ms)"], row["Test accuracy"]),
                 textcoords="offset points", xytext=(8, 4))
plt.xlabel("Inference time per image (ms, GPU)")
plt.ylabel("Test accuracy")
plt.title("Accuracy vs speed (bubble size = model size)")
plt.grid(alpha=0.3)
plt.savefig(FIG_DIR / "accuracy_vs_speed.png", bbox_inches="tight")
plt.show()

## 10. K-fold cross-validation (best model)

A single train/validation split can be lucky or unlucky. **K-fold cross-validation** checks how stable
a model is:

1. The train + validation images (85% of the data) are divided into **K = 5 equal parts (folds)**.
2. The model is trained 5 times. Each time, a different fold is held out for evaluation and the other 4 are
   used for training.
3. The 5 results are averaged. A **small standard deviation** means the result does not depend on which
   images happened to be in the validation set.

```
Fold 1:  [eval] [train] [train] [train] [train]
Fold 2:  [train] [eval] [train] [train] [train]
Fold 3:  [train] [train] [eval] [train] [train]
Fold 4:  [train] [train] [train] [eval] [train]
Fold 5:  [train] [train] [train] [train] [eval]
```

`StratifiedKFold` keeps the cat/dog balance in every fold. The **test set is not used here**.
Training 5 times is expensive, so it is done only for the best model and with fewer epochs
(3 + 2 instead of 5 + 5), with a fixed number of epochs and no early stopping (the evaluation fold is not
used during training at all). Each fold is saved to Drive as soon as it finishes.

In [ ]:
cv_df = pd.concat([df_train, df_val], ignore_index=True)
skf = StratifiedKFold(n_splits=KFOLDS, shuffle=True, random_state=SEED)
cv_file = RESULTS_DIR / f"kfold_{BEST}.json"
folds = json.loads(cv_file.read_text()) if cv_file.exists() else []

for fold, (train_idx, eval_idx) in enumerate(skf.split(cv_df["filename"], cv_df["label"]), start=1):
    if fold <= len(folds):
        print(f"Fold {fold}: already done")
        continue
    print(f"===== {BEST}: fold {fold}/{KFOLDS} =====")
    keras.backend.clear_session()
    gc.collect()
    fold_train = make_dataset(cv_df.iloc[train_idx], training=True)
    fold_eval = make_dataset(cv_df.iloc[eval_idx], training=False)

    model, base = MODEL_SPECS[BEST]["build"]()
    compile_model(model, 1e-3)
    head = KFOLD_HEAD_EPOCHS if base is not None else KFOLD_HEAD_EPOCHS + KFOLD_FINE_TUNE_EPOCHS
    model.fit(fold_train, epochs=head, verbose=2)
    if base is not None:
        unfreeze_top(base, FINE_TUNE_FRACTION)
        compile_model(model, 1e-5)
        model.fit(fold_train, epochs=head + KFOLD_FINE_TUNE_EPOCHS, initial_epoch=head, verbose=2)

    y_true = np.concatenate([y.numpy() for _, y in fold_eval]).astype(int)
    y_prob = model.predict(fold_eval, verbose=0).ravel()
    metrics = {"fold": fold, **compute_metrics(y_true, y_prob)}
    folds.append(metrics)
    cv_file.write_text(json.dumps(folds, indent=2))
    print({k: round(v, 4) for k, v in metrics.items() if k in metric_names})
    del fold_train, fold_eval, model, base

In [ ]:
kfold_df = pd.DataFrame(folds).set_index("fold")[metric_names]
kfold_summary = pd.DataFrame({"mean": kfold_df.mean(), "std": kfold_df.std()})
print(f"{KFOLDS}-fold cross-validation of {BEST}")
display(kfold_df.style.format("{:.4f}"))
display(kfold_summary.style.format("{:.4f}"))

ax = kfold_summary["mean"].plot.bar(yerr=kfold_summary["std"], capsize=6, figsize=(8, 4), rot=0, color="#666666")
ax.set_ylim(max(0.0, kfold_summary["mean"].min() - 0.05), 1.0)
ax.set_title(f"{BEST}: {KFOLDS}-fold cross-validation (mean +- std)")
plt.savefig(FIG_DIR / "kfold.png", bbox_inches="tight")
plt.show()

## 11. Predictions of the best model on test images

In [ ]:
best_model = keras.models.load_model(MODELS_DIR / f"{BEST}.keras")
images, labels = next(iter(test_ds))
probs = best_model.predict(images, verbose=0).ravel()

plt.figure(figsize=(14, 10))
for i in range(min(12, len(images))):
    plt.subplot(3, 4, i + 1)
    plt.imshow(images[i].numpy().astype("uint8"))
    pred = CLASS_NAMES[int(probs[i] >= 0.5)]
    conf = max(probs[i], 1 - probs[i])
    ok = pred == CLASS_NAMES[int(labels[i])]
    plt.title(f"{pred} ({conf:.1%})", color="green" if ok else "red")
    plt.axis("off")
plt.suptitle(f"{BEST} predictions on test images (red = wrong)")
plt.savefig(FIG_DIR / "test_predictions.png", bbox_inches="tight")
plt.show()

## 12. Export the best model for the backend

The exported model is saved **without the optimizer state** (only needed for training), which makes the file
much smaller. `model_info.json` tells the backend the image size and class names, and keeps the comparison
and cross-validation results.

In [ ]:
MODEL_FILE = "cats_dogs_model.keras"
for old in EXPORT_DIR.glob("*"):
    shutil.rmtree(old) if old.is_dir() else old.unlink()

export_model = keras.models.load_model(MODELS_DIR / f"{BEST}.keras", compile=False)
export_model.save(EXPORT_DIR / MODEL_FILE)
shutil.copytree(FIG_DIR, EXPORT_DIR / "figures")
shutil.copytree(SPLITS_DIR, EXPORT_DIR / "splits")
shutil.copy(RESULTS_DIR / "comparison.csv", EXPORT_DIR / "comparison.csv")

best = results[BEST]
info = {
    "model_file": MODEL_FILE,
    "architecture": BEST,
    "img_size": IMG_SIZE,
    "class_names": CLASS_NAMES,
    "output": "single sigmoid unit = probability of class_names[1] (Dog)",
    "input": "RGB float32 in [0, 255], bilinear resize to img_size; normalization is inside the model",
    "selection": "highest validation F1",
    "test_accuracy": best["test"]["accuracy"],
    "baseline_test_accuracy": results["BaselineCNN"]["test"]["accuracy"],
    "test_metrics": best["test"],
    "validation_metrics": best["validation"],
    "kfold": {"k": KFOLDS, "mean": kfold_summary["mean"].to_dict(), "std": kfold_summary["std"].to_dict()},
    "comparison": json.loads(comparison.to_json(orient="records")),
    "dataset_sizes": {"train": len(df_train), "val": len(df_val), "test": len(df_test),
                      "per_class": int(len(cats)), "balanced": True},
    "tensorflow_version": tf.__version__,
    "keras_version": keras.__version__,
    "trained_at": time.strftime("%Y-%m-%d %H:%M:%S"),
}
(EXPORT_DIR / "model_info.json").write_text(json.dumps(info, indent=2))
print(json.dumps({k: v for k, v in info.items() if k not in ("comparison", "test_metrics", "validation_metrics")}, indent=2))
print(f"Exported model size: {(EXPORT_DIR / MODEL_FILE).stat().st_size / 1e6:.1f} MB")

### Sanity check
Reload the exported file and predict raw image files exactly the way the Flask backend does.

In [ ]:
reloaded = keras.models.load_model(EXPORT_DIR / MODEL_FILE)
for path, label in df_test.head(6).itertuples(index=False):
    img = tf.io.decode_jpeg(tf.io.read_file(path), channels=3)
    img = tf.image.resize(img, (IMG_SIZE, IMG_SIZE))
    p = float(reloaded.predict(img[tf.newaxis, ...], verbose=0)[0, 0])
    print(f"{path:30s} true={CLASS_NAMES[label]:3s}  P(dog)={p:.4f}")

In [ ]:
shutil.make_archive("cats_dogs_export", "zip", EXPORT_DIR)
print("Created cats_dogs_export.zip (also saved in Drive:", EXPORT_DIR, ")")
try:
    from google.colab import files
    files.download("cats_dogs_export.zip")
except ImportError:
    pass